# NUMPY MASTERY — CHAPTERS 23–26

```
NUMPY MASTERY
Current Chapters: 23 (Matrix Operations), 24 (Linear Systems),
                  25 (Eigenvalues/Eigenvectors), 26 (SVD)
Progress:         26 / 75 chapters
Completed:        Ch1-22
Remaining:        49 chapters
```

This is the core `np.linalg` toolkit, grouped as one continuous story: matrix properties
(23) → solving equations (24) → understanding a matrix's fundamental structure via eigen-
decomposition (25) → the most general and powerful decomposition, SVD (26), which underlies
PCA (Chapter 42), recommendation systems (Chapter 49), and image compression.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 23 — Matrix Operations

## Concept
Beyond multiplication, `np.linalg` exposes the core scalar/structural properties of a matrix:
**rank** (how many independent directions of information it contains), **determinant** (a
scalar measuring volume-scaling / invertibility), **inverse** (the matrix that "undoes" it),
and **norms** (magnitude measures for vectors and matrices).

## Why it matters
Rank tells you if your feature matrix has redundant (linearly dependent) columns. Determinant
near zero signals a matrix is close to singular (numerically dangerous to invert). Norms
appear everywhere: regularization terms (L2 norm of weights), gradient clipping (norm of the
gradient vector), distance metrics (Chapter 41, KNN).


In [ ]:
A = np.array([[4., 2.],
              [1., 3.]])

print("A:\n", A)
print("A.T (transpose):\n", A.T)
print()
print("np.linalg.matrix_rank(A):", np.linalg.matrix_rank(A))
print("np.linalg.det(A)        :", np.linalg.det(A))
print("np.linalg.inv(A):\n", np.linalg.inv(A))
print()
print("Verify A @ inv(A) == Identity:\n", np.round(A @ np.linalg.inv(A), 10))


### Expected Output
```
A:
 [[4. 2.]
  [1. 3.]]
A.T (transpose):
 [[4. 1.]
  [2. 3.]]

np.linalg.matrix_rank(A): 2
np.linalg.det(A)        : 10.000000000000002
np.linalg.inv(A):
 [[ 0.3 -0.2]
  [-0.1  0.4]]

Verify A @ inv(A) == Identity:
 [[ 1.  0.]
 [-0.  1.]]
```

### Rank, geometrically and practically
`matrix_rank(A) == 2` for this `2x2` matrix means both rows (and both columns) are linearly
independent — the matrix uses the full 2D space it lives in. If a feature matrix has
`rank < n_features`, some features are exact linear combinations of others (perfect
multicollinearity) — this causes numerical instability in linear regression's normal equations
and is worth checking during EDA: `np.linalg.matrix_rank(X)` vs `X.shape[1]`.

### Determinant intuition
`det(A) ≈ 10` here — geometrically, a unit square transformed by `A` becomes a parallelogram
with area `|det(A)|`. `det(A) == 0` means the transformation collapses space into a lower
dimension (information is lost, not invertible). `det(A)` very close to (but not exactly) zero
signals a matrix that's technically invertible but numerically unstable to invert — small
input changes cause huge output changes (Chapter 27, numerical stability).

### `np.linalg.norm()` — vector and matrix magnitudes


In [ ]:
v = np.array([3., 4.])
M = np.array([[1., 2.], [3., 4.]])

print("L2 norm of v (default)         :", np.linalg.norm(v), "  <- sqrt(3^2+4^2) = 5, the familiar 'Euclidean length'")
print("L1 norm of v                    :", np.linalg.norm(v, ord=1), "  <- sum(|v_i|) = 3+4 = 7")
print("Infinity norm of v               :", np.linalg.norm(v, ord=np.inf), "  <- max(|v_i|) = 4")
print()
print("Frobenius norm of matrix M       :", np.linalg.norm(M), "  <- sqrt(sum of all squared elements)")
print("Row-wise L2 norms of a batch     :", np.linalg.norm(np.array([[3,4],[6,8]]), axis=1), "  <- axis param, just like Ch.10!")


### Expected Output
```
L2 norm of v (default)         : 5.0   <- sqrt(3^2+4^2) = 5, the familiar 'Euclidean length'
L1 norm of v                    : 7.0   <- sum(|v_i|) = 3+4 = 7
Infinity norm of v               : 4.0   <- max(|v_i|) = 4

Frobenius norm of matrix M       : 5.477225575051661   <- sqrt(sum of all squared elements)
Row-wise L2 norms of a batch     : [ 5. 10.]   <- axis param, just like Ch.10!
```

### Real-world use
- **L2 (Euclidean) norm**: distance between points (Chapter 41, KNN), and is the basis of
  Ridge/L2 regularization (penalize `norm(weights)^2`).
- **L1 norm**: basis of Lasso/L1 regularization (encourages sparse weight vectors — many exact
  zeros), and is more robust to outliers than L2.
- **Gradient clipping in deep learning**: `if norm(gradient) > threshold: gradient *=
  threshold / norm(gradient)` — a direct, common use of `np.linalg.norm`.
- **`axis` parameter on `norm`**: computing the L2 norm of every row in a `(n_samples,
  n_features)` matrix in one vectorized call — exactly the axis-collapsing reasoning from
  Chapter 10, applied here to distances.

### Why explicit matrix inversion is usually NOT the production approach


In [ ]:
import time

rng = np.random.default_rng(0)
n = 500
A_big = rng.normal(size=(n, n))
b_big = rng.normal(size=n)

t0 = time.perf_counter()
x_via_inverse = np.linalg.inv(A_big) @ b_big
t1 = time.perf_counter()

t2 = time.perf_counter()
x_via_solve = np.linalg.solve(A_big, b_big)
t3 = time.perf_counter()

print(f"Via explicit inverse : {t1 - t0:.5f} s")
print(f"Via np.linalg.solve  : {t3 - t2:.5f} s")
print("Results match?", np.allclose(x_via_inverse, x_via_solve))


### Expected Output (timings vary by machine, but `solve` is reliably faster — verified on NumPy 2.4.4)
```
Via explicit inverse : 0.11215 s
Via np.linalg.solve  : 0.00773 s   (here roughly 14.5x faster, and more numerically stable)
Results match? True
```

### Why avoid explicit inversion (this connects directly into Chapter 24)
Computing `inv(A)` explicitly and then multiplying by `b` does strictly more work than
necessary (a full matrix inversion, an `O(n^3)` operation, followed by a matrix-vector
multiply) compared to directly solving the linear system `Ax = b` via specialized
decomposition-based algorithms (LU decomposition under the hood) that `np.linalg.solve` uses.
Beyond raw speed, explicit inversion tends to **accumulate more numerical error**, especially
for ill-conditioned matrices (near-zero determinant) — a classic "textbook says one thing,
production code does another" case. **Rule: if you need to solve `Ax=b`, use `solve()`; only
compute an explicit inverse if you genuinely need the inverse matrix itself for something else
(rare in practice).**

### Interview Insight
> "Why should you avoid `np.linalg.inv(A) @ b` in favor of `np.linalg.solve(A, b)`?"
`solve()` uses a more efficient and numerically stable decomposition-based algorithm tailored
to solving the specific system, avoiding the unnecessary extra work and accumulated error of
computing a full explicit inverse you don't actually need.


---
# CHAPTER 24 — Linear Systems

## Concept
`np.linalg.solve(A, b)` solves the exact system `Ax = b` for square, full-rank `A`.
`np.linalg.lstsq(A, b)` solves the **least-squares** problem — the best approximate solution
when the system is over-determined (more equations than unknowns, the standard regression
setup) or under-determined, and `A` need not even be square.

## Why it matters
This is the mathematical foundation of linear regression. "Fit a line/hyperplane to data" is
*literally* "find the least-squares solution to an over-determined linear system."


In [ ]:
# --- Exact square system ---
A = np.array([[3., 1.],
              [1., 2.]])
b = np.array([9., 8.])

x = np.linalg.solve(A, b)
print("Solving Ax = b for square A:")
print("x =", x)
print("Verify A @ x == b:", np.allclose(A @ x, b))


### Expected Output
```
Solving Ax = b for square A:
x = [2. 3.]
Verify A @ x == b: True
```


In [ ]:
# --- Building linear regression from mathematical first principles ---
rng = np.random.default_rng(42)
n_samples = 100

true_slope, true_intercept = 2.5, 1.0
x_data = rng.uniform(0, 10, n_samples)
noise = rng.normal(0, 1, n_samples)
y_data = true_slope * x_data + true_intercept + noise

# Design matrix: add a column of 1s for the intercept term
X_design = np.column_stack([x_data, np.ones(n_samples)])   # shape (100, 2)
print("X_design.shape:", X_design.shape, " (n_samples, [slope_feature, intercept_feature])")

# Solve the least-squares problem: find [slope, intercept] minimizing sum of squared residuals
coeffs, residuals, rank, singular_values = np.linalg.lstsq(X_design, y_data, rcond=None)
fitted_slope, fitted_intercept = coeffs

print(f"\nTrue slope={true_slope}, intercept={true_intercept}")
print(f"Fitted slope={fitted_slope:.4f}, intercept={fitted_intercept:.4f}")
print(f"Design matrix rank: {rank}  (full rank = 2, meaning both columns are independent)")


### Expected Output
```
X_design.shape: (100, 2) (n_samples, [slope_feature, intercept_feature])

True slope=2.5, intercept=1.0
Fitted slope=2.5077, intercept=0.9487
Design matrix rank: 2  (full rank = 2, meaning both columns are independent)
```

### What just happened, mathematically
`np.linalg.lstsq` found the `[slope, intercept]` vector that **minimizes the sum of squared
residuals** `||X_design @ coeffs - y_data||^2` — this is *exactly* the definition of ordinary
least-squares linear regression. The `np.column_stack([x_data, np.ones(n_samples)])` trick
(adding a constant column of 1s) is the standard way to fold the intercept term into a single
matrix equation, so `X_design @ [slope, intercept]` computes `slope*x + intercept*1` for every
row at once.

### `solve` vs `lstsq` — when to use which
| | `np.linalg.solve(A, b)` | `np.linalg.lstsq(A, b)` |
|---|---|---|
| Requires `A` square? | Yes | No |
| Requires exact solution to exist? | Yes (errors if singular) | No — finds best approximate fit |
| Typical use | Exact systems (rare in ML) | Regression, over-determined systems (common in ML) |

### Common Mistake
Using `np.linalg.solve()` on a non-square, over-determined design matrix (more samples than
features, the typical regression setup) — this raises an error, because `solve` requires a
square system. This is a very common early-ML-engineer confusion; the fix is always `lstsq`
(or, in practice, a dedicated regression implementation like scikit-learn's, which uses more
numerically robust internals but solves the same underlying problem).

### Interview Insight
> "Mathematically, what is linear regression doing under the hood?"
It's finding the least-squares solution to the (generally over-determined, since n_samples >
n_features) linear system `X @ w ≈ y` — i.e., the weight vector `w` that minimizes the sum of
squared residuals `||Xw - y||^2`. `np.linalg.lstsq` (or, more numerically robust in practice,
an SVD-based approach — see Chapter 26) solves exactly this problem.


---
# CHAPTER 25 — Eigenvalues and Eigenvectors

## Concept
For a square matrix `A`, an **eigenvector** `v` is a direction that `A` doesn't rotate — it
only stretches or shrinks it: `A @ v = λ * v`, where `λ` (the **eigenvalue**) is the scaling
factor. `np.linalg.eig()` computes both for a general square matrix; `np.linalg.eigh()` is a
specialized, faster, more numerically stable version for **symmetric** matrices (which is what
covariance matrices always are).

## Why it matters
PCA (Chapter 42) works by finding the eigenvectors of a data's covariance matrix — the
directions of maximum variance in the data. This chapter is the direct mathematical
prerequisite for that.


In [ ]:
A = np.array([[4., 1.],
              [2., 3.]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print("A:\n", A)
print("\neigenvalues :", eigenvalues)
print("eigenvectors (as COLUMNS):\n", eigenvectors)

# Verify the defining property: A @ v == lambda * v, for each eigenvector/eigenvalue pair
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    print(f"\nEigenpair {i}: lambda={lam:.4f}")
    print("  A @ v        :", A @ v)
    print("  lambda * v   :", lam * v)
    print("  match?", np.allclose(A @ v, lam * v))


### Expected Output
```
A:
 [[4. 1.]
  [2. 3.]]

eigenvalues : [5. 2.]
eigenvectors (as COLUMNS):
 [[ 0.70710678 -0.4472136 ]
  [ 0.70710678  0.89442719]]

Eigenpair 0: lambda=5.0000
  A @ v        : [3.53553391 3.53553391]
  lambda * v   : [3.53553391 3.53553391]
  match? True

Eigenpair 1: lambda=2.0000
  A @ v        : [-0.89442719  1.78885438]
  lambda * v   : [-0.89442719  1.78885438]
  match? True
```

### Critical gotcha: eigenvectors are the **columns**, not the rows
`eigenvectors[:, i]` is the i-th eigenvector, corresponding to `eigenvalues[i]`. Accidentally
using `eigenvectors[i, :]` (a row) is one of the most common `eig()` bugs — it silently gives
you a *different*, meaningless vector instead of raising an error.

### Why `eigh()` matters for symmetric matrices (like covariance matrices)


In [ ]:
rng = np.random.default_rng(0)
data = rng.normal(size=(200, 4))
cov_matrix = np.cov(data, rowvar=False)   # (4,4) covariance matrix -- ALWAYS symmetric

print("Is cov_matrix symmetric?", np.allclose(cov_matrix, cov_matrix.T))

import time
t0 = time.perf_counter()
for _ in range(2000):
    np.linalg.eig(cov_matrix)
t1 = time.perf_counter()

t2 = time.perf_counter()
for _ in range(2000):
    np.linalg.eigh(cov_matrix)
t3 = time.perf_counter()

print(f"\neig()  total time (2000 calls): {t1-t0:.4f} s")
print(f"eigh() total time (2000 calls): {t3-t2:.4f} s")

eigvals_eig, _ = np.linalg.eig(cov_matrix)
eigvals_eigh, _ = np.linalg.eigh(cov_matrix)
print("\neig()  eigenvalues :", np.sort(eigvals_eig.real))
print("eigh() eigenvalues :", eigvals_eigh, " <- eigh() ALWAYS returns them sorted ascending, and guaranteed real")


### Expected Output (timings vary by machine; eigh should be noticeably faster — verified on NumPy 2.4.4)
```
Is cov_matrix symmetric? True

eig()  total time (2000 calls): 0.0570 s
eigh() total time (2000 calls): 0.0210 s   (here roughly 2.7x faster for symmetric input)

eig()  eigenvalues : [0.89357274 0.92374746 1.077142   1.10845476]
eigh() eigenvalues : [0.89357274 0.92374746 1.077142   1.10845476]   <- eigh() ALWAYS returns them sorted ascending, and guaranteed real
```

### Why `eigh` specifically for symmetric matrices
For a general matrix, `eig()` uses a slower, more general algorithm and can return **complex**
eigenvalues/eigenvectors even when the "meaningful" answer is real (numerical noise can
introduce tiny imaginary parts). For symmetric matrices, a mathematical guarantee holds:
eigenvalues are always real, and eigenvectors are always orthogonal. `eigh()` exploits this
guarantee with a specialized, faster, more numerically stable algorithm, and always returns
clean real eigenvalues sorted in ascending order — no `.real` post-processing needed. Since
covariance matrices are *always* symmetric, PCA implementations should always use `eigh()`, not
`eig()`.

### Interview Insight
> "You're implementing PCA and need the eigendecomposition of a covariance matrix. Would you
> use `eig()` or `eigh()`, and why?"
`eigh()` — covariance matrices are always symmetric, and `eigh()` exploits that structure for a
faster, more numerically stable computation, with guaranteed real, ascending-sorted
eigenvalues, avoiding the complex-number noise `eig()` can introduce on the same input.


---
# CHAPTER 26 — Singular Value Decomposition (SVD)

## Concept
SVD factors **any** matrix `A` (not just square ones) into three matrices:
```
A = U @ Σ @ Vᵀ
```
- `U`: orthogonal matrix (left singular vectors) — shape `(m, m)`
- `Σ`: diagonal matrix of **singular values**, sorted descending — shape `(m, n)` (returned as
  a 1D array `s` by NumPy, containing just the diagonal)
- `Vᵀ`: orthogonal matrix (right singular vectors, transposed) — shape `(n, n)`

## Why it matters
SVD is the single most important decomposition in applied linear algebra for ML: it powers
PCA (an SVD of the centered data matrix is a more numerically stable way to do PCA than
eigendecomposing the covariance matrix directly), low-rank approximation (image/data
compression), and the mathematical foundation of many recommendation systems (matrix
factorization).


In [ ]:
A = np.array([[3., 1., 1.],
              [-1., 3., 1.]])   # shape (2, 3) -- SVD works on NON-square matrices too!

U, s, Vt = np.linalg.svd(A)
print("A.shape:", A.shape)
print("U.shape:", U.shape, " (left singular vectors)")
print("s.shape:", s.shape, " (singular values, 1D -- the diagonal of Sigma)")
print("Vt.shape:", Vt.shape, " (right singular vectors, transposed)")
print("\nsingular values:", s, " <- always sorted descending")

# Reconstruct A from U, s, Vt to verify the decomposition
Sigma = np.zeros((U.shape[0], Vt.shape[0]))
np.fill_diagonal(Sigma, s)
A_reconstructed = U @ Sigma @ Vt
print("\nReconstructed A:\n", A_reconstructed)
print("Matches original A?", np.allclose(A, A_reconstructed))


### Expected Output
```
A.shape: (2, 3)
U.shape: (2, 2)  (left singular vectors)
s.shape: (2,)  (singular values, 1D -- the diagonal of Sigma)
Vt.shape: (3, 3)  (right singular vectors, transposed)

singular values: [3.46410162 3.16227766]  <- always sorted descending

Reconstructed A:
 [[ 3.  1.  1.]
  [-1.  3.  1.]]
Matches original A?  True
```

### Low-rank approximation — the practical payoff
Because singular values are sorted descending, keeping only the **top k** singular
values/vectors gives the best possible rank-k approximation of `A` (in a precise mathematical
sense — minimizes reconstruction error among all rank-k matrices). This is the basis of image
compression and dimensionality reduction.


In [ ]:
# --- Practical image-compression-style demo using a synthetic "image" matrix ---
rng = np.random.default_rng(0)
# Build a LOW-RANK-ish synthetic image: outer products of a few patterns + a little noise
h, w = 60, 80
pattern1 = np.outer(np.sin(np.linspace(0, 3, h)), np.cos(np.linspace(0, 2, w)))
pattern2 = np.outer(np.linspace(-1, 1, h), np.linspace(1, -1, w))
image = 3 * pattern1 + 1.5 * pattern2 + 0.05 * rng.normal(size=(h, w))

U, s, Vt = np.linalg.svd(image, full_matrices=False)
print("image.shape:", image.shape, " full singular value count:", len(s))
print("Top 10 singular values:", np.round(s[:10], 3))
print("(notice how quickly they drop off -- most information is in the first few)")

for k in [1, 2, 5, 20]:
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    error = np.linalg.norm(image - approx) / np.linalg.norm(image)
    original_size = h * w
    compressed_size = k * (h + w + 1)   # elements needed to store U[:, :k], s[:k], Vt[:k, :]
    print(f"k={k:3d}: relative reconstruction error={error:.4f}   "
          f"compression ratio={compressed_size/original_size:.3f} "
          f"({compressed_size} vs {original_size} numbers)")


### Expected Output (verified on NumPy 2.4.4 for this exact synthetic image construction)
```
image.shape: (60, 80)  full singular value count: 60
Top 10 singular values: [100.313  24.228   0.787   0.785   0.756   0.708   0.69    0.671   0.664
   0.643]
(notice how quickly they drop off -- most information is in the first few)

k=  1: relative reconstruction error=0.2369   compression ratio=0.029 (141 vs 4800 numbers)
k=  2: relative reconstruction error=0.0324   compression ratio=0.059 (282 vs 4800 numbers)
k=  5: relative reconstruction error=0.0296   compression ratio=0.147 (705 vs 4800 numbers)
k= 20: relative reconstruction error=0.0188   compression ratio=0.588 (2820 vs 4800 numbers)
```

### Reading this result
Because our synthetic image was constructed from just 2 underlying patterns plus a little
noise, `k=2` already captures nearly all the structure (very low reconstruction error) while
storing only ~6% of the original number of values — this is the essence of SVD-based
compression: real images/datasets often have most of their "information" concentrated in a
small number of singular values/vectors, letting you discard the rest with minimal quality
loss.

### SVD's connection to PCA (fully developed in Chapter 42)
PCA on a centered data matrix `X` (mean-subtracted, shape `(n_samples, n_features)`) can be
computed either by eigendecomposing the covariance matrix `X.T @ X` (Chapter 25's approach), or
by taking the SVD of `X` directly — the right singular vectors `V` from `X`'s SVD are exactly
the principal component directions, and the singular values relate directly to the amount of
variance each component explains (`variance_explained = s**2 / (n_samples - 1)`). The
SVD-based approach is generally preferred in production because it's more numerically stable —
it avoids ever explicitly forming `X.T @ X`, which can amplify numerical errors (squaring
values can lose precision for ill-conditioned data), directly connecting back to Chapter 23's
"avoid unnecessary matrix operations" lesson.

### Interview Insight
> "How does SVD relate to PCA, and why might you prefer computing PCA via SVD over eigen-
> decomposing the covariance matrix directly?"
The right singular vectors of the (centered) data matrix are the principal component
directions; singular values relate directly to explained variance. SVD is often preferred in
production because it avoids explicitly forming `X.T @ X`, which can be numerically less
stable (squaring the data can amplify floating-point error, especially for ill-conditioned or
high-dimensional data) — a direct instance of the general "avoid unnecessary matrix operations
for numerical stability" principle.


---
## Chapters 23–26 Summary

- **Matrix operations**: `matrix_rank` reveals redundant/dependent features; `det` near zero
  signals numerical danger; `norm` (with `ord` and `axis` parameters) powers regularization,
  gradient clipping, and distance computations; prefer `solve()` over explicit `inv()` — it's
  faster and more numerically stable for solving `Ax=b`.
- **Linear systems**: `solve()` requires square, exactly-solvable systems; `lstsq()` handles the
  realistic ML case (over-determined, approximate best-fit) — literally *is* linear regression's
  mathematical definition (minimize `||Xw - y||^2`).
- **Eigenvalues/eigenvectors**: `A @ v = λv`; eigenvectors live in the **columns** of `eig()`'s
  output, a common indexing trap. `eigh()` should always be used for symmetric matrices
  (covariance matrices always qualify) — faster, more stable, guaranteed real, sorted results.
- **SVD**: `A = U @ Σ @ Vᵀ` works on any matrix shape; sorted-descending singular values enable
  optimal low-rank approximation (compression); SVD is the numerically preferred route to PCA,
  avoiding the instability of explicitly forming `X.T @ X`.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For a `(5, 5)` matrix `A` with `np.linalg.matrix_rank(A) == 3`,
what does this tell you about `np.linalg.det(A)`, and about attempting `np.linalg.inv(A)`?

**Exercise 2:** You have a `(1000, 50)` feature matrix `X` and target vector `y` of shape
`(1000,)`. Which `np.linalg` function directly gives you the least-squares regression
coefficients, and what shape design matrix modification do you need first to also fit an
intercept term?

**Coding Challenge:** Implement `pca_via_svd(X, n_components)` that centers `X` (subtracts
per-feature mean), computes its SVD, and returns the data projected onto the top
`n_components` principal directions — without eigendecomposing any covariance matrix. Compare
the explained variance ratio you compute against `s**2 / (n_samples - 1)`, normalized to sum to
1.

**Debugging Challenge:**
```python
A = np.array([[2., 0.], [0., 3.]])
eigenvalues, eigenvectors = np.linalg.eig(A)
first_eigenvector = eigenvectors[0]   # a teammate assumes this is the FIRST eigenvector
# but np.allclose(A @ first_eigenvector, eigenvalues[0] * first_eigenvector) is False!
```
Explain the indexing bug and the one-character fix.

**Interview Questions:**
1. Why is `np.linalg.solve(A, b)` generally preferred over `np.linalg.inv(A) @ b`?
2. What's the difference between `solve()` and `lstsq()`, and which one is relevant for linear
   regression, and why?
3. Why should `eigh()` be used instead of `eig()` for covariance matrices specifically?
4. Explain, at a high level, how SVD enables low-rank approximation / compression.

**Quiz:**
- Q1: True/False — `np.linalg.eig()` on a symmetric matrix can, due to floating point noise,
  occasionally return complex eigenvalues even though the true mathematical answer is real.
- Q2: In `U, s, Vt = np.linalg.svd(A)`, is `s` returned as a 1D array or a full diagonal 2D
  matrix by default?
- Q3: What single number (0 to 1) would you compute to check "how much information is lost" by
  a rank-k SVD approximation, using the reconstructed vs original matrix?

---

